# 5주차 랩 · 오류의 비용이 바뀌면 판정 기준은 어떻게 달라질까

**⏱ 60분** · 교재 5-1·5-2와 같은 Scania APS 기록·분할·모델 · 개인 제출

## 🎯 이번 랩의 질문

교재에서는 오경보 10, 미탐지 500의 비용으로 검증에서 판정 기준을 골랐습니다. 그런데 오경보를 처리하는 데 드는 비용이 더 크거나, APS 고장을 놓쳤을 때의 손실이 더 크다면 기준도 달라져야 합니다.

> **오경보 또는 미탐지가 비싸지면 APS 판정 기준을 어떻게 바꿔야 할까요?**

교재에서 반복한 패턴을 비용 조건 네 가지에 다시 씁니다.

| 패턴 | 교재 | 이번 랩 |
|---|---|---|
| 판정 → 두 오류 세기 → 비용 | 5-1 2·3절 | 과제 1·2 |
| 후보마다 비용 → 가장 싼 기준(동률이면 가장 큰 기준) | 5-2 2·3절 | 과제 3·4 |
| 고정한 기준을 테스트에서 평가 | 5-2 6절 | 과제 5·6 |
| 숫자를 근거로 판단 쓰기 | 5-2 7절 | 과제 7 |

비용 조건은 `COSTS`에 (오경보 단가, 미탐지 단가) 순서로 (10, 500), (50, 500), (100, 500), (10, 5000)을 담았습니다. 첫 조건은 원자료의 평가 비용이고, 나머지는 비용이 바뀌는 경우를 비교하려는 가정입니다. 맞힌 판정의 비용은 0입니다. 실제 수리비나 예방 효과를 계산하는 과제는 아닙니다.

### 진행 방법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행합니다 |
| 📝 | 과제 | `None` 자리를 코드로 바꿔 채웁니다. 변수 이름은 바꾸지 않습니다 |
| ✅ | 확인 셀 | 고치지 말고 실행만 합니다 |
| 💬 | 결과 읽기 | 내 결과에서 무엇을 볼지 짚습니다. 과제 7에서 이 숫자들을 씁니다 |

- 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. 첫 실행에는 자료 읽기와 모델 학습에 1~2분이 더 걸릴 수 있습니다. 결과는 반올림하지 않고 변수에 저장합니다.
- **✅ 확인 셀의 "형식 확인 완료"는 정답이라는 뜻이 아닙니다.** 작성 여부·자료형·표 구조만 봅니다. 계산값은 제출 후 교수자가 채점하고, 과제 7의 서술은 따로 읽습니다.
- 시간: 준비 5분 · 과제 1~3 각 5분 · 과제 4 10분 · 과제 5 10분 · 과제 6 5분 · 과제 7 15분

## ▶ 준비

교재 5-1 3·4절(분할·중앙값·모델·검증 점수)과 5-2 1·3절(`errors` 함수, 모든 후보의 두 오류)을 같은 설정으로 실행합니다.

- `p_val`은 검증 점수, `y_val`은 검증 정답입니다. `errors(정답, 판정)`은 (오경보, 미탐지)를 돌려줍니다.
- `curve`는 모든 검증 후보의 `threshold`·`fp`·`fn`·`positive`(APS로 판정한 수)를 담고, 기준 오름차순입니다. **비용 열은 아직 없습니다.** 이 표에서 단가만 바꿔 비용을 계산합니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 교재 5-1 3·4절, 5-2 1·3절과 같은 설정입니다. (1~2분)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import load
aps = load("aps")
tr = aps["train"]
te = aps["test"]
from balab import checker, summary, check_format
check = checker("week05")
X = tr.drop(columns=["class"])
y = (tr["class"] == "pos").astype(int)
X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=12000, stratify=y, random_state=0)
med = X_fit.median()
X_fit = X_fit.fillna(med)
X_val = X_val.fillna(med)
model = LGBMClassifier(
    num_leaves=15, learning_rate=0.03, n_estimators=800,
    min_child_samples=20, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, verbose=-1, random_state=0,
    n_jobs=1, deterministic=True, force_col_wise=True)
model.fit(X_fit, y_fit)
p_val = pd.Series(model.predict_proba(X_val)[:, 1], index=X_val.index)

def errors(actual, predicted):
    fp = int(((predicted == 1) & (actual == 0)).sum())
    fn = int(((predicted == 0) & (actual == 1)).sum())
    return fp, fn

thresholds = [0.0] + sorted(p_val.unique()) + [float("inf")]
rows = []
for threshold in thresholds:
    predicted = (p_val >= threshold).astype(int)
    fp, fn = errors(y_val, predicted)
    rows.append([threshold, fp, fn, int(predicted.sum())])
curve = pd.DataFrame(rows, columns=["threshold", "fp", "fn", "positive"])
COSTS = [(10, 500), (50, 500), (100, 500), (10, 5000)]
curve.head()

✅ `curve`의 앞 다섯 행이 나오면 됩니다. 첫 행(기준 0)은 전원 "APS" 판정이라 오경보가 11,800건입니다.

---
## 📝 과제 1 · 기본값 0.5에서 두 오류는 몇 건인가

비교의 출발점은 흔히 쓰는 기준 0.5입니다. 교재 5-2 3절 반복문 안의 **판정 → 두 오류** 두 줄을 기준 0.5로 한 번 씁니다.

- `p_val`이 0.5 이상이면 1, 아니면 0인 판정을 `called_half`에 담습니다.
- `errors`에 `y_val`과 판정을 넣어 오경보를 `fp_half`, 미탐지를 `fn_half`에 받습니다.

**결과 형식:** `fp_half`, `fn_half`는 정수

In [ ]:
# 📝 과제 1 — None 자리를 채우세요
# 힌트: 교재 5-2 3절 반복문 안의 판정 한 줄과 errors 한 줄에서 기준 자리에 0.5를 씁니다.
called_half = None  # 검증 점수 0.5 이상이면 APS 판정(1)
fp_half = None      # 기본값의 오경보 수
fn_half = None      # 기본값의 미탐지 수

In [ ]:
# ✅ 확인 — 과제 1 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(1, fp_half=fp_half, fn_half=fn_half)
else:
    check_format(1, fp_half=fp_half, fn_half=fn_half)

💬 **결과 읽기** — 두 오류 중 어느 쪽이 더 많나요? 미탐지 한 건이 오경보 50건과 같은 값이라는 점을 떠올리며 과제 2의 비용을 봅니다.

---
## 📝 과제 2 · 기본값과 두 극단의 검증 비용

오경보 10, 미탐지 500을 적용합니다.

- 과제 1의 두 오류로 기본값의 검증 비용 `cost_half_val`을 구합니다.
- 교재 5-1 3절처럼 전원 "APS 아님"(`y_val * 0`)과 전원 "APS"(`y_val * 0 + 1`) 판정의 비용을 `cost_none_val`, `cost_all_val`에 담습니다. 전원 "APS 아님"이면 APS 기록이 모두 미탐지이고, 전원 "APS"이면 나머지가 모두 오경보입니다.

**결과 형식:** 정수 세 개

In [ ]:
# 📝 과제 2 — None 자리를 채우세요
# 힌트: 비용은 10 × 오경보 + 500 × 미탐지입니다. 두 극단은 교재 5-1 3절의 두 줄과 같습니다.
cost_half_val = None  # 기본값 0.5의 검증 비용
cost_none_val = None  # 전원 'APS 아님'의 검증 비용
cost_all_val = None   # 전원 'APS'의 검증 비용

In [ ]:
# ✅ 확인 — 과제 2 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(2, cost_half_val=cost_half_val, cost_none_val=cost_none_val, cost_all_val=cost_all_val)
else:
    check_format(2, cost_half_val=cost_half_val, cost_none_val=cost_none_val, cost_all_val=cost_all_val)

💬 **결과 읽기** — 기본값 0.5의 비용을 두 극단과, 교재 5-2 3절에서 고른 기준의 검증 비용과 비교해 보세요.

---
## 📝 과제 3 · 기본 비용 조건에서 가장 싼 기준

교재 5-2 3절과 같은 선택을 직접 해 봅니다.

- `curve`의 `fp`·`fn`에 단가 10·500을 곱해 `cost` 열을 만듭니다.
- 교재 5-2 2절처럼 최소 비용 행을 `loc`으로 남기고 `iloc`의 -1로 가장 큰 기준을 고릅니다. 고른 행을 `best_default`에, 그 행의 `threshold`와 `cost`를 `threshold_default`, `cost_default`에 담습니다.
- 테스트 정답은 쓰지 않습니다.

**결과 형식:** `threshold_default`, `cost_default`는 실수

In [ ]:
# 📝 과제 3 — None 자리를 채우세요
# 힌트: 교재 5-2 3절의 비용 열 만들기 한 줄과 가장 싼 행 고르기 한 줄을 씁니다.
best_default = None       # 기본 비용에서 고른 후보 행
threshold_default = None  # 고른 기준
cost_default = None       # 고른 기준의 검증 비용

In [ ]:
# ✅ 확인 — 과제 3 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if threshold_default is None or cost_default is None:
        check(3, threshold_default=None)
    else:
        check(3, threshold_default=(threshold_default, 0.000001), cost_default=cost_default)
else:
    check_format(3, threshold_default=threshold_default, cost_default=cost_default)

💬 **결과 읽기** — 교재 5-2 3절의 결과와 같은 기준·비용이 나왔는지 확인합니다.

---
## 📝 과제 4 · 비용 조건마다 가장 싼 기준

과제 3의 계산을 `COSTS`의 네 조건에서 반복합니다. 모델과 분할은 바꾸지 않고, 같은 `curve`에서 비용 열만 조건마다 다시 계산합니다.

- 교재 5-2 3절처럼 `for`로 조건을 하나씩 꺼내 `rows`에 `append`합니다. 한 행은 [오경보 단가, 미탐지 단가, 고른 기준, 검증 비용]입니다.
- 결과는 길이 4의 목록입니다.

In [ ]:
# 📝 과제 4 (1) — None 자리를 채우세요
# 힌트: COSTS에서 두 단가를 한 쌍씩 꺼내는 for 반복을 만들고, 반복 안에서 과제 3의 두 줄을 단가만 바꿔 씁니다.
rows = None  # 조건별 [단가 두 개, 고른 기준, 검증 비용]을 모은 목록

모은 결과를 비용 조건별 표로 만듭니다.

- `rows`를 DataFrame으로 바꿔 `selected`에 담고, 열 이름은 `c_fp`·`c_fn`·`threshold`·`val_cost` 순서로 줍니다.

**결과 형식:** 4행 4열 표, 행 순서는 `COSTS`와 같음

In [ ]:
# 📝 과제 4 (2) — None 자리를 채우세요
# 힌트: 교재 5-2 3절에서 rows로 curve를 만든 것과 같은 방법입니다. 열 이름만 네 개로 바꿉니다.
selected = None  # rows를 네 열로 만든 선택 표

In [ ]:
# ✅ 확인 — 과제 4 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if selected is None or not {"threshold", "val_cost"}.issubset(selected.columns):
        check(4, selected=None)
    else:
        check(4, selected=selected, chosen_thresholds=(selected["threshold"].tolist(), 0.000001), chosen_costs=(selected["val_cost"].tolist(), 1.0))
else:
    check_format(4, selected=selected)

💬 **결과 읽기** — 첫 행이 과제 3과 같은지 먼저 봅니다. 오경보 단가를 올린 두 조건(50, 100)과 미탐지 단가를 올린 조건(5000)에서 기준은 각각 어느 쪽으로 움직였나요? 비싸진 오류를 줄이는 쪽으로 움직였는지 확인합니다.

---
## 📝 과제 5 · 고른 기준을 테스트에서 평가

교재 5-2 6절처럼 처음 보는 테스트 기록에 적용합니다. 아래 ▶ 셀이 테스트 점수와 정답을 준비합니다. 학습용 중앙값·모델·고른 기준은 바꾸지 않습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 테스트 입력에도 학습용 중앙값과 고정한 모델을 적용합니다 (교재 5-2 6절)
X_te = te.drop(columns=["class"]).fillna(med)
y_te = (te["class"] == "pos").astype(int)
p_te = pd.Series(model.predict_proba(X_te)[:, 1], index=X_te.index)

- `selected`의 행을 `range(len(selected))`로 하나씩 꺼내고(`selected.iloc[i]`), 그 행의 기준으로 `p_te`를 판정합니다.
- `errors`로 두 오류를 세고, **그 행의 단가**로 비용을 계산합니다.
- 한 행은 [`c_fp`, `c_fn`, `threshold`, `fp`, `fn`, `positive`, `test_cost`]입니다. `positive`는 APS로 판정한 수입니다. `test_rows`에 모읍니다.

In [ ]:
# 📝 과제 5 (1) — None 자리를 채우세요
# 힌트: 교재 5-2 6절 반복문과 같은 구조입니다. 판정 기준과 두 단가는 selected의 각 행에서 꺼냅니다.
test_rows = None  # 조건별 테스트 판정 결과를 모은 목록

`test_rows`를 DataFrame으로 바꿔 `evaluated`에 담습니다. 열 이름과 순서는 위의 일곱 항목입니다.

**결과 형식:** 4행 7열 표

In [ ]:
# 📝 과제 5 (2) — None 자리를 채우세요
# 힌트: 과제 4 (2)와 같은 방법입니다. 열 이름만 일곱 개로 바꿉니다.
evaluated = None  # 테스트 결과를 일곱 열로 만든 표

In [ ]:
# ✅ 확인 — 과제 5 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if evaluated is None or not {"test_cost", "fp", "fn"}.issubset(evaluated.columns):
        check(5, evaluated=None)
    else:
        check(5, evaluated=evaluated, test_costs=(evaluated["test_cost"].tolist(), 1.0), test_fp=(evaluated["fp"].tolist(), 1.0), test_fn=(evaluated["fn"].tolist(), 1.0))
else:
    check_format(5, evaluated=evaluated)

💬 **결과 읽기** — 단가가 다른 행끼리 총비용을 바로 비교하지 않습니다. 행마다 기준에 따라 오경보·미탐지·APS 판정 수가 어떻게 달라졌는지 읽습니다.

---
## 📝 과제 6 · 같은 단가의 기본값과 비교

교재 5-2 6절 마지막 셀처럼, 고른 기준의 비용을 **같은 단가로 계산한** 기본값 0.5의 비용으로 나눕니다.

- 테스트 점수에 0.5를 적용해 두 오류를 셉니다.
- `COSTS`의 각 단가로 기본값의 테스트 비용을 `half_test_costs`에 모읍니다.
- 같은 조건의 `evaluated` 비용을 그 값으로 나눈 비율을 `cost_ratios`에 모읍니다.

**결과 형식:** 길이 4의 목록 두 개, 순서는 `COSTS`와 같음

In [ ]:
# 📝 과제 6 — None 자리를 채우세요
# 힌트: 0.5 판정의 두 오류는 한 번만 세고, for로 COSTS를 돌며 단가만 바꿔 비용과 비율을 append합니다.
half_test_costs = None  # 비용 조건별 기본값 0.5의 테스트 비용 목록
cost_ratios = None      # 같은 조건에서 고른 기준의 비용 ÷ 기본값 비용

In [ ]:
# ✅ 확인 — 과제 6 (고치지 말고 실행만 하세요)
if os.environ.get("BALAB_ANSWER_KEY"):
    if half_test_costs is None or cost_ratios is None:
        check(6, half_test_costs=None)
    else:
        check(6, half_test_costs=(half_test_costs, 1.0), cost_ratios=(cost_ratios, 0.001))
else:
    check_format(6, half_test_costs=half_test_costs, cost_ratios=cost_ratios)

💬 **결과 읽기** — 비율이 1보다 작으면 이 테스트에서는 고른 기준의 비용이 기본값보다 낮다는 뜻입니다. 어느 조건에서 비율이 가장 작고, 왜 그럴까요?

---
## 📝 과제 7 · 본인의 판단과 설명

계산 결과를 근거로 본인의 판단을 씁니다. 사용한 수치를 함께 적습니다.

1. 오경보 단가를 올린 조건과 미탐지 단가를 올린 조건을 하나씩 골라, 기준·두 오류·APS 판정 수가 어떻게 달라졌는지 설명합니다.
2. 같은 단가의 기본값 대비 비용 비율을 근거로 기준 선택을 평가합니다.
3. 이 결과의 적용 범위를 씁니다. 단가가 다른 총비용을 직접 비교하는 것과, 줄어든 미탐지를 실제로 예방한 고장으로 읽는 것에 대해 본인의 판단을 씁니다. 테스트 결과를 본 뒤 기준을 다시 고르지 않습니다.

자동 채점은 하지 않습니다.

✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

---
## ✅ 제출 전 확인

아래 셀을 실행해 과제 1~6의 작성 여부와 형식을 확인합니다. "형식 확인 완료"는 계산값이 맞다는 뜻이 아닙니다. 결과를 직접 읽고 과제 7을 작성해 제출합니다. 실제 값 채점은 제출 후 진행합니다.

In [ ]:
# ✅ 제출 전 확인 — 과제 1~6의 작성 여부와 형식
summary()